# Query Enhancement – Query Expansion Techniques

In a RAG pipeline, the quality of the query sent to the retriever determines how good the retrieved context is—and therefore, how accurate the LLM's final answer will be.

That's where Query Expansion / Enhancement comes in.

## 🎯 What is Query Enhancement?

Query enhancement refers to techniques used to improve or reformulate the user query to retrieve better, more relevant documents from the knowledge base.

It is especially useful when:

- The original query is short, ambiguous, or under-specified.
- You want to broaden the scope to catch synonyms, related phrases, or spelling variants.

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

In [5]:
# Step 1: Load the dataset and split the data

loader = TextLoader("langchain_crewai_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(raw_docs)


In [6]:
chunks

[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [10]:
## Step 2 : Creste a vectorStore
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorStore = FAISS.from_documents(chunks,embedding_model)

# Step 3 : create a Retriever --> MMR
retriever = vectorStore.as_retriever(search_type="mmr",search_kwargs={"k":5})
retriever



Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7608.28it/s]


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000292A3B95D30>, search_type='mmr', search_kwargs={'k': 5})

In [11]:
# step 4 :  LLM and Prompt
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


llm = init_chat_model(
    "llama-3.3-70b-versatile",
    model_provider="groq"
)

In [12]:
# Query Enhancement(expansion) Prompt 
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonym, technical terms and useful context.

Original query: "{query}"

Expanded query:
""")

query_expansion_chain = query_expansion_prompt | llm | StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonym, technical terms and useful context.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatGroq(metadata={'versions': {'langchain-core': '1.4.6', 'langchain': '1.3.8'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000292973E5160>, async_client=<groq.resources.chat.completions.AsyncCompleti

In [17]:
query_expansion_chain.invoke({"query":"Langchain memory"})

'Expanded query: "Langchain memory" OR "Langchain knowledge retention" OR "Large language model memory" OR "Conversational AI recall" OR "LLaMA memory capabilities" OR "Language model information storage" OR "Chatbot memory architecture" OR "Natural language processing memory management" OR "Transformer-based language model knowledge retention" OR "Artificial intelligence memory optimization"\n\nAdditionally, you can also consider adding relevant technical terms and context such as:\n\n* "Working memory" OR "long-term memory" to specify the type of memory being referred to\n* "Attention mechanisms" OR "memory-augmented neural networks" to include relevant architectural components\n* "Knowledge graph" OR "knowledge embedding" to include relevant data structures and techniques\n* "Question answering" OR "conversational dialogue" to include relevant applications and use cases\n* "Efficient memory usage" OR "memory optimization techniques" to include relevant performance considerations\n\n

In [18]:
# RAG answering Prompt
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

document_chain = create_stuff_documents_chain(llm=llm,prompt=answer_prompt)

In [19]:
# Full rag pipelinr with query expansion

rag_pipeline =(
    RunnableMap({
        "input": lambda x: x["input"],
        "context": lambda x: retriever.invoke(query_expansion_chain.invoke({"query": x["input"]}))
    })
    | document_chain
)

In [21]:
# Step 6: Run query

query = {"input": "What types of memory does LangChain support?"}

print(query_expansion_chain.invoke({"query": query}))

response = rag_pipeline.invoke(query)

print("✅ Answer:\n", response)

Expanded query: 
{
    'input': 'What types of memory does LangChain support, including short-term memory, long-term memory, episodic memory, semantic memory, working memory, and external memory? Are there any specific memory architectures, such as attention-based memory or graph-based memory, that are utilized in LangChain? How does LangChain handle memory management, memory retrieval, and memory updating, particularly in the context of large language models, conversational AI, and natural language processing applications?'
}
✅ Answer:
 LangChain supports two types of memory modules: 

1. ConversationBufferMemory: This allows the LLM to maintain awareness of previous conversation turns.
2. ConversationSummaryMemory: This allows the LLM to summarize long interactions to fit within token limits.


In [27]:
# Step 6: Run query

query = {"input": "CrewAI Agents"}

print(query_expansion_chain.invoke({"query": query}))

response = rag_pipeline.invoke(query)

print("✅ Answer:\n", response)

Expanded query: 
{
    'input': [
        'CrewAI Agents', 
        'Artificial Intelligence Crew Members', 
        'AI-Powered Crew Systems', 
        'Autonomous Crew Agents', 
        'Intelligent Crew Assistants', 
        'Machine Learning-Based Crew Solutions', 
        'Crew Automation Platforms', 
        'AI-Driven Crew Management', 
        'Cognitive Crew Agents', 
        'Human-Machine Interface Crew Systems'
    ]
}

This expanded query incorporates relevant synonyms, technical terms, and useful context to improve document retrieval. It includes various phrases that might be used to describe CrewAI Agents, such as artificial intelligence, machine learning, autonomy, and human-machine interface. This should help retrieve a wider range of relevant documents and information related to CrewAI Agents.
✅ Answer:
 Based on the provided context, CrewAI agents have the following characteristics:

1. **Defined Role**: Each agent in a crew has a defined role, such as researcher, pl